# Unidad 14. Herencia y polimorfismo

**Duración orientativa:** 7 horas · **Versión:** 2.0 · **UT2**

La herencia permite especializar una clase existente. El polimorfismo permite usar objetos diferentes mediante una misma operación. Ambas técnicas solo aportan claridad cuando modelan una relación real.

## Resultados de aprendizaje

Al finalizar podrás identificar una relación de especialización, crear subclases, reutilizar estado con super, redefinir métodos respetando un contrato, aplicar polimorfismo y decidir cuándo es preferible composición.

## 1. Antes de heredar: es un

La herencia expresa que una subclase es una forma especializada de la base. SensorTemperatura es un Sensor; por eso debe poder utilizarse donde se espera un Sensor. Un Aula tiene sensores, pero no es un Sensor: esa relación se expresa por composición.

No heredes para reutilizar unas pocas líneas. Pregunta primero si la frase X es un Y es verdadera de forma natural. Si no lo es, usa composición o extrae una función.

## 2. Clase base y subclase

La base contiene estado y operaciones comunes. La subclase hereda esos elementos, añade lo específico y puede redefinir un método. La base debe ser pequeña y expresar una interfaz común, sin conocer todos los subtipos futuros.

Una interfaz es un acuerdo: si todos los sensores tienen leer, el resto del programa puede solicitar lecturas sin comprobar clases concretas. Las subclases pueden variar internamente, pero deben devolver resultados compatibles.

In [1]:
class Sensor:
    def __init__(self, codigo: str, activo: bool = True) -> None:
        self.codigo = codigo
        self.activo = activo

    def descripcion(self) -> str:
        return f"Sensor {self.codigo}, activo={self.activo}"

class SensorTemperatura(Sensor):
    pass

sensor = SensorTemperatura("T-01")
print(sensor.descripcion())


Sensor T-01, activo=True


## 3. Constructor y super

Super llama el comportamiento de la clase base. En el constructor evita copiar la inicialización de atributos compartidos. Primero se inicializa el estado común; después se añaden atributos propios. Si la base valida un código, todas las subclases comparten esa validación.

Copiar el constructor en cada subclase crea duplicación y errores cuando la regla cambia. Olvidar super suele dejar atributos comunes sin crear; es uno de los primeros puntos que revisar al depurar una jerarquía.

In [2]:
class SensorTemperatura(Sensor):
    def __init__(self, codigo: str, valor: float, activo: bool = True) -> None:
        super().__init__(codigo, activo)
        self.valor = valor

    def leer(self) -> dict[str, object]:
        if not self.activo:
            return {"codigo": self.codigo, "estado": "inactivo"}
        return {"codigo": self.codigo, "tipo": "temperatura", "valor": self.valor, "unidad": "°C"}

print(SensorTemperatura("T-02", 21.5).leer())


{'codigo': 'T-02', 'tipo': 'temperatura', 'valor': 21.5, 'unidad': '°C'}


## 4. Redefinir métodos y mantener el contrato

Una subclase puede redefinir leer, pero quien procesa sensores necesita saber qué estructura recibirá. En este cuaderno todas las lecturas activas devuelven codigo, tipo, valor y unidad; las inactivas devuelven un estado. Es una decisión de diseño que debe documentarse y probarse.

No uses una larga cadena de comprobaciones de tipo para seleccionar cada caso. Si cada subtipo conoce cómo leer su propia medida, el objeto adecuado toma la decisión. Eso es polimorfismo.

In [3]:
class SensorHumedad(Sensor):
    def __init__(self, codigo: str, valor: float) -> None:
        super().__init__(codigo)
        self.valor = valor

    def leer(self) -> dict[str, object]:
        return {"codigo": self.codigo, "tipo": "humedad", "valor": self.valor, "unidad": "%"}

sensores = [SensorTemperatura("T-03", 22.0), SensorHumedad("H-01", 44.0)]
for sensor in sensores:
    print(sensor.leer())


{'codigo': 'T-03', 'tipo': 'temperatura', 'valor': 22.0, 'unidad': '°C'}
{'codigo': 'H-01', 'tipo': 'humedad', 'valor': 44.0, 'unidad': '%'}


## 5. Polimorfismo y duck typing

El bucle anterior solicita leer a cada objeto sin preguntar su tipo. Cada instancia responde con su método. Añadir SensorCO2 no debería exigir modificar el bucle, solo crear una clase que respete el acuerdo. Python usa también duck typing: importa que un objeto ofrezca la operación necesaria, no solo su clase exacta.

Prueba primero cada subtipo por separado y después una lista mezclada. Un fallo común es que una subclase devuelva una estructura distinta y el error aparezca más tarde al preparar el informe.

## 6. Herencia frente a composición

La composición expresa tiene un: una Estacion contiene sensores y un Informe contiene lecturas. Es flexible y evita jerarquías artificiales. Usa herencia para una especialización estable con interfaz común.

Señales para evitar herencia: una subclase anula muchos métodos, necesita excepciones constantes o no se puede usar como la base. En esos casos, composición y funciones pequeñas suelen resultar más sencillas y comprobables.

## Práctica guiada

1. Crea dos instancias de cada subtipo y comprueba que su estado es independiente.
2. Haz temporalmente inactivo un sensor de temperatura y predice su resultado.
3. Añade ubicacion a la base y comprueba que ambas subclases la reciben con super.
4. Clasifica Aula-Sensor, Sensor-SensorCO2, Informe-Medicion y Vehiculo-Bicicleta como herencia o composición.
5. Explica qué modificarías para añadir SensorCO2 sin cambiar el bucle polimórfico.

## Ejercicios

**Ejercicio 14.01.** Crea Dispositivo con codigo y descripcion. Especializa SensorLuz y SensorRuido redefiniendo descripcion.

**Ejercicio 14.02.** Define mostrar_descripciones que recorra dispositivos y llame a descripcion sin comprobar tipos. Explica qué parte es polimórfica.

**Ejercicio 14.03.** Crea una base Sensor con leer y los subtipos SensorTemperatura y SensorHumedad. Garantiza el mismo formato de diccionario y prueba tipo, unidad e inactividad.

In [8]:
# Ejercicio 14.01
class Dispositivo:
    def __init__(self, codigo, descripcion):
        self.codigo = codigo
        self.descripcion = descripcion

class SensorLuz(Dispositivo):
    def __init__(self, codigo):
        super().__init__(codigo, "Sensor de Luz")

class SensorRuido(Dispositivo):
    def __init__(self, codigo):
        super().__init__(codigo, "Sensor de Ruido")

# Ejercicio 14.02
def mostrar_descripciones(dispositivos: list):
    for dispositivo in dispositivos:
        print(f"Descripcion {dispositivo.descripcion}")

# Ejercicio 14.03 Crea una base Sensor con leer y los subtipos SensorTemperatura y SensorHumedad. Garantiza el mismo formato de diccionario y prueba tipo, unidad e inactividad.
class Sensor:
    def __init__(self, activo: bool = True):
        self.activo = activo

    def leer(self):
        pass

class SensorTemperatura(Sensor):

    def leer(self) -> dict:
        if not self.activo:
            return {
                "tipo": "temperatura",
                "unidad": "°C",
                "activo": self.activo
            }
        return {
            "tipo": "temperatura",
            "unidad": "°C",
        }

class SensorHumedad(Sensor):
    def leer(self) -> dict:
        if not self.activo:
            return {
                "tipo": "humedad",
                "unidad": "%",
                "activo": self.activo
            }
        return {
            "tipo": "humedad",
            "unidad": "%",
        }

temperatura = SensorTemperatura()
humedad = SensorHumedad()

# Tipo
assert temperatura.leer()["tipo"] == "temperatura"
assert humedad.leer()["tipo"] == "humedad"

# Unidad
assert temperatura.leer()["unidad"] == "°C"
assert humedad.leer()["unidad"] == "%"

# Inactividad
sensor_inactivo = SensorTemperatura(False)
assert sensor_inactivo.activo is False

## Proyecto 14 · Red extensible de sensores

Implementa Sensor con codigo y activo, y SensorTemperatura, SensorHumedad y SensorCO2. Cada subtipo devolverá mediante leer un diccionario compatible. Implementa generar_informe con un bucle que acepte una lista mezclada sin comprobar clases concretas.

Incluye docstrings, anotaciones y pruebas para una lectura de cada subtipo, un sensor inactivo, una lista mezclada y la incorporación de SensorCO2 sin modificar generar_informe. Documenta qué comportamiento comparten, por qué Estacion tendría sensores por composición y una situación en que no usarías herencia.

In [22]:
class Sensor:
    def __init__(self, codigo: str, activo: bool):
        self.codigo = codigo
        self.activo = activo

class SensorTemperatura(Sensor):
    def __init__(self, codigo: str, activo: bool, temperatura : float):
        super().__init__(codigo, activo)
        self.temperatura = temperatura

    def leer(self) -> dict:
        """Devuelve un diccionario con los datos de la clase"""
        if self.activo:
            return {
                "Codigo": self.codigo,
                "Activo": self.activo,
                "Tipo": "temperatura",
                "Valor": self.temperatura
            }
        else:
            return {}

class SensorHumedad(Sensor):
    def __init__(self, codigo: str, activo: bool, humedad : float):
        super().__init__(codigo, activo)
        self.humedad = humedad

    def leer(self) -> dict:
        """Devuelve un diccionario con los datos de la clase"""
        if self.activo:
            return {
                "Codigo": self.codigo,
                "Activo": self.activo,
                "Tipo": "humedad",
                "Valor": self.humedad
            }
        else:
            return {}

class SensorCO2(Sensor):
    def __init__(self, codigo: str, activo: bool, co2 : float):
        super().__init__(codigo, activo)
        self.co2= co2

    def leer(self) -> dict:
        """Devuelve un diccionario con los datos de la clase"""
        if self.activo:
            return {
                "Codigo": self.codigo,
                "Activo": self.activo,
                "Tipo": "CO2",
                "Valor": self.co2
            }
        else:
            return {}

def generar_informe(lista: list) -> list:
    """Genera un listado con todos los diccionarios"""
    listado_de_informes = []

    for sensor in lista:
        lectura = sensor.leer()
        if lectura:
            listado_de_informes.append(lectura)

    return listado_de_informes

sensor1 = SensorTemperatura("A-11", True, 22.5)
sensor2 = SensorHumedad("A-12", True, 55)
sensor3 = SensorCO2("A-13", True, 155)
sensor_inactivo = SensorCO2("A-13", False, 155)
lista_mezclada = [sensor1, sensor2, sensor3, sensor_inactivo]

assert sensor1.leer()["Tipo"] == "temperatura"
assert sensor2.leer()["Tipo"] == "humedad"
assert sensor3.leer()["Tipo"] == "CO2"

assert SensorCO2.leer(sensor_inactivo) == {}

assert len(generar_informe(lista_mezclada)) == 3






## Comportamiento compartido
- Todos los sensores tienen un codigo, si están activos y un metodo leer()

## Por qué Estación tendría sensores por composición
- Una estacion contiene varios sensores. Por lo que la composicion es más correcto.

# No usaría herencia cuando no comparten mismos atributos o cuando cada clase tenga métodos totalmente distintos

## Autoevaluación

¿Puedes justificar es un, explicar super, describir el contrato de leer, explicar polimorfismo y elegir composición cuando una entidad contiene otra?

- [Herencia en Python](https://docs.python.org/3/tutorial/classes.html#inheritance)